# SynthID Remover on Kaggle
Run with **Accelerator: GPU** and **Internet: On** (Settings panel).
Models (~21 GB) and ComfyUI are downloaded to **scratch disk** (not `/kaggle/working`, which is capped at ~20 GB and saved as output).

In [ ]:
# 1. Pick the scratch dir with the most free space.
# Kaggle docs/sources disagree on the scratch path name (/kaggle/temp vs /kaggle/tmp), so probe them.
import os, shutil
candidates = ["/kaggle/temp", "/kaggle/tmp", "/tmp"]
best, best_free = None, -1
for c in candidates:
    try:
        os.makedirs(c, exist_ok=True)
        free = shutil.disk_usage(c).free / 1e9
        print(f"{c}: {free:.1f} GB free")
        if free > best_free: best, best_free = c, free
    except Exception as e:
        print(f"{c}: unusable ({e})")
SCRATCH = os.path.join(best, "synth")
os.makedirs(SCRATCH, exist_ok=True)
assert best_free >= 30, f"Only {best_free:.0f} GB free in {best}; need ~30 GB (models ~21 GB + deps)"
print("Using scratch:", SCRATCH)

# Send every large download to scratch
os.environ["SYNTH_DATA_DIR"] = SCRATCH               # ComfyUI + models (see app.py)
os.environ["HF_HOME"] = f"{SCRATCH}/hf_home"           # Hugging Face cache
os.environ["PIP_CACHE_DIR"] = f"{SCRATCH}/pip_cache"
os.environ["TORCH_HOME"] = f"{SCRATCH}/torch_home"
os.environ["SKIP_PERF_OPTS"] = "1"                     # skip flash-attn source build (very slow on Kaggle)

In [ ]:
# 2. Get the app code (small, fine in /kaggle/working)
import subprocess
REPO = "https://github.com/de3sec/synth-id-remover"
APP = "/kaggle/working/synth-id-remover"
shutil.rmtree(f"{APP}/ComfyUI", ignore_errors=True)  # never keep models in /kaggle/working
if not os.path.exists(APP):
    subprocess.run(["git", "clone", REPO, APP], check=True)
else:  # existing clone (maybe from an older repo): repoint and sync
    subprocess.run(["git", "remote", "set-url", "origin", REPO], cwd=APP, check=True)
    subprocess.run(["git", "fetch", "-q", "origin"], cwd=APP, check=True)
    subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], cwd=APP, check=True)
os.chdir(APP)


In [ ]:
# 3. Dependencies. Keep Kaggle's preinstalled CUDA torch; install the rest + ComfyUI's own requirements.
# ComfyUI is pinned to v0.4.0 (v0.3.77 lacks the Z-Image Fun ControlNet loader; versions from 2026-01-19 break dype): the pinned ComfyUI-dype node breaks on newer ComfyUI (pe_embedder -> rope_embedder rename).
COMFYUI_REF = "v0.4.0"
!grep -vE '^(torch|torchvision|torchaudio)$' requirements.txt > /kaggle/working/req_notorch.txt
!pip install -q -r /kaggle/working/req_notorch.txt
if not os.path.exists(f"{SCRATCH}/ComfyUI"):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", f"{SCRATCH}/ComfyUI"], check=True)
subprocess.run(["git", "fetch", "--tags", "-q"], cwd=f"{SCRATCH}/ComfyUI", check=True)
subprocess.run(["git", "checkout", "-q", "-f", COMFYUI_REF], cwd=f"{SCRATCH}/ComfyUI", check=True)  # -f drops the local MIG patches (app.py re-applies them); untracked models are kept
!pip install -q -r {SCRATCH}/ComfyUI/requirements.txt


In [ ]:
# 4. First-time setup: clones custom nodes and downloads ~21 GB of models into scratch. Takes a while.
import app            # runs setup() on import
assert app.COMFYUI_DIR.startswith(SCRATCH), app.COMFYUI_DIR
!du -sh {SCRATCH}/ComfyUI/models/* ; df -h {SCRATCH}

In [ ]:
# 5a. Run on an image (upload it via "Add Input" as a dataset, or put a path here)
from PIL import Image
IMG = "/kaggle/input/YOUR_DATASET/image.png"   # <-- change me
result = app.remove_watermark(Image.open(IMG).convert("RGB"))
result.save("/kaggle/working/output.png")      # small; kept as notebook output
result

In [ ]:
# 5b. Or launch the Gradio UI with a public share link
os.environ["GRADIO_SHARE"] = "1"
app.demo.launch(css=app.css, share=True)

**Notes**
- Scratch disk is wiped when the session ends, so models re-download each session (~21 GB).
- P100/T4 GPUs lack native bf16; the bf16 weights may run slowly or need more VRAM. Pick the GPU option with the most VRAM available.
- The 120 s processing timeout in `app.py` may be too short on Kaggle GPUs.